# Behavioral questions

## Tell me about a time your AI system didn’t work or gave wrong answers.

In my turbine RAG project, the system initially gave incorrect answers for some troubleshooting queries. I investigated and found the issue was mainly in retrieval quality.

on the retrieval side, we initially used ANN-based search, but later moved to a more precise KNN approach to improve result accuracy. This helped improve the relevance of retrieved chunks.

Top-k retrieved chucks calcueted there average search score and if the retrievev chucks score is less than avarea it will ingored.

On the generation side, I improved the prompt to strictly restrict the model to the provided context, which helped reduce hallucinations. I also implemented confidence filtering—if the retrieval score was below a certain threshold, the system returned a fallback instead of generating a potentially incorrect answer.

As a result, incorrect answers reduced by around 40%, and the system became much more reliable and trustworthy for engineers.

###  Why move from ANN to KNN?

ANN was faster but sometimes returned less accurate neighbors. Since our use case required higher precision for engineering queries, we switched to KNN to improve retrieval accuracy, even with a slight latency trade-off.

## Why did you choose RAG instead of fine-tuning (or vice versa)?

In my use case, I chose RAG over fine-tuning primarily because the data was dynamic and frequently updated, such as turbine manuals and troubleshooting documents. With RAG, we can retrieve the latest information at query time without retraining the model.

Fine-tuning, on the other hand, is better suited for learning patterns or behavior, but it’s static and requires retraining whenever data changes, which is costly and time-consuming.

RAG also provides better transparency since we can show the retrieved sources, which is important for engineering use cases where trust and explainability matter.

So overall, RAG gave us flexibility, lower cost, and more reliable, up-to-date answers compared to fine-tuning.”

###  If They Ask Follow-Up: “When WOULD you use fine-tuning?”

“I would use fine-tuning when the problem requires learning consistent patterns or behavior, like response formatting, tone control, classification, or domain-specific reasoning that retrieval alone cannot handle.”

## How did you reduce latency or improve performance?

To improve latency in my RAG system, I optimized both retrieval and generation. On the retrieval side, I tuned the top-k value and controlled candidate size to balance speed and relevance, especially after moving from ANN to KNN.

On the generation side, I reduced prompt size by passing only the most relevant chunks, which lowered token usage and response time. I also added confidence filtering to avoid unnecessary LLM calls for low-quality queries.

Overall, this reduced unnecessary processing while maintaining answer quality and improving system responsiveness.

### What specific latency improvements did you see?”

“We reduced unnecessary LLM calls for low-confidence queries and optimized retrieval size, which helped improve overall response time while keeping accuracy high.”

### What would you do further to reduce latency?”

“I would add caching for frequent queries, use smaller or faster models where possible, and introduce asynchronous or streaming responses to improve perceived latency.”

## How do you evaluate your AI system?

“I evaluate at two levels—retrieval and generation. For retrieval, I check relevance of top-k results. For generation, I validate correctness and grounding while monitoring hallucinations. I also use manual testing with real queries and compare performance before and after changes.

##  How did you deploy your AI system?
I deployed the AI system using a containerized approach with Docker. The FastAPI backend, which handled the RAG pipeline, was packaged into a container and deployed on Azure App Service.

For CI/CD, I used a pipeline to automatically build and deploy the application whenever changes were pushed, ensuring smooth and consistent releases.

Secrets like API keys and connection strings were managed securely using Azure Key Vault and referenced through managed identity, so nothing was hardcoded.

The system was integrated with Azure AI Search for retrieval and an LLM for generation, and we added logging and monitoring to track performance and errors after deployment.

Overall, this setup ensured scalability, security, and easy maintainability in production.”

### How does the request flow work?”

“User → Frontend → FastAPI API → Retrieval (Azure AI Search) → LLM → Response → User”

### “How did you handle scaling?”

“Azure App Service handled auto-scaling, and since the API is stateless, it scales horizontally easily.”

## Tell me about a data-related issue you faced

I faced an issue where unstructured documents led to poor retrieval quality. I fixed it by improving preprocessing and chunking, making chunks more focused. This significantly improved answer accuracy.”

### “What exactly was wrong with the data?”

“The data had mixed topics in the same sections, so chunks were not semantically clean.”

### “What did you learn?”
“Better data structuring directly improves retrieval, which is critical for RAG performance.”

## Tell me about a trade-off you made (cost vs accuracy, speed vs quality)

In my RAG system, I had to make a trade-off between speed and accuracy during retrieval. Initially, we used ANN-based search, which was fast but sometimes returned less precise results, leading to lower answer quality.

Since our use case involved engineering troubleshooting, accuracy was more critical than speed. So I switched to a KNN-based approach, which improved retrieval precision but increased latency by around 15–20%.

To balance this, I optimized top-k retrieval and added confidence filtering to avoid unnecessary LLM calls for low-quality queries.

This allowed us to prioritize answer quality while keeping latency within an acceptable range

### How did you decide the trade-off?”

“Based on user impact—engineers needed reliable answers more than fast but incorrect ones.”

## Your chatbot is giving wrong answers. What will you do?”

If my chatbot is giving wrong answers, I would debug it systematically across the RAG pipeline.

First, I would check retrieval quality—whether the top-k chunks actually contain relevant information. If not, I’d look at issues like chunking, search strategy, or embeddings.

Second, I’d validate the prompt to ensure the model is strictly grounded in the provided context and not hallucinating.

Third, I’d analyze the retrieved content itself—if the data is noisy or poorly structured, that can lead to incorrect answers.

I’d also check confidence scores and add or tune fallback mechanisms to avoid answering when retrieval is weak.

Finally, I’d use test queries and logs to compare before and after changes to ensure the fixes are actually improving accuracy.

Overall, I focus on identifying whether the issue is in retrieval, data, or generation, and fix it step by step.

## Tell me about a time requirements were unclear. What did you do?

When requirements were unclear, I clarified them by sharing sample outputs and getting feedback from stakeholders. This helped refine expectations and build the system correctly from the start.

### What did you learn?”
“Early feedback is critical, especially in AI systems where expectations can vary widely.”

## System worked fine initially but failed at scale. What happened?

Initially, the system worked well during testing, but when usage increased, we started seeing performance degradation and inconsistent response quality.

After analyzing the issue, I found two main causes. First, the retrieval layer wasn’t optimized for scale—higher query volume increased latency, especially after moving to a more precise KNN-based search. Second, we were sending too many chunks to the LLM, which increased response time and token usage under load.

To fix this, I optimized the top-k retrieval to limit unnecessary data, added stricter filtering to ensure only relevant chunks were passed, and introduced confidence-based early exits to reduce unnecessary LLM calls. I also ensured the API was stateless so it could scale horizontally.

After these changes, the system handled higher load more efficiently with improved response consistency.

## Tell me about a time you had to coordinate across different technical areas or teammates with different expertise. How did you ensure everyone stayed aligned?

I aligned cross-functional teams by defining clear API contracts, sharing sample outputs including edge cases, and maintaining regular sync-ups. This ensured smooth integration and avoided miscommunication.


## Describe a time you had to bridge the gap between a technical solution you built and a non-technical stakeholder. How did you translate the system's behavior into something meaningful for them?

I explained the AI system as a ‘smart search + summarization tool’ instead of using technical terms, and used real examples to show its behavior. This helped non-technical stakeholders understand and trust the system.


### What was the biggest challenge?
“Explaining why the system sometimes doesn’t answer—people expect AI to always respond.


## Describe a real debugging scenario where a silent failure caused unexpected behavior — like Azure AI Search returning fewer than K results — and how you built a safeguard.

I encountered a silent failure where Azure AI Search returned fewer than top-k results. I added safeguards by checking result count and average similarity score, and triggered fallback responses when confidence was low, preventing incorrect answers.

## Tell me about a time you encountered a subtle bug like a cosine similarity false positive. What was your debugging process?

“In my RAG system, I encountered a subtle issue where cosine similarity was returning false positives—chunks that were semantically similar at a high level but not actually relevant to the user’s query. This caused the model to generate partially incorrect answers even though retrieval scores looked high.

To debug this, I first logged the top-k retrieved chunks along with their similarity scores and manually inspected them. I noticed that some chunks shared common keywords or general context but didn’t answer the specific query.

I then validated this by testing with multiple queries and confirmed that high similarity didn’t always mean high relevance.

To fix it, I introduced stricter filtering using an average similarity threshold and limited the number of chunks passed to the model. I also improved the prompt to focus on precise context.

This reduced false positives and improved answer accuracy significantly.

This experience taught me that similarity scores alone aren’t enough—you need additional validation to ensure true relevance

### What would you do further?
Add re-ranking (cross-encoder) or metadata filtering to improve precision.